# Day 3 牛津 Tutorial LLM 仿真 (v6.0 学习科学层)

> **配套**: v5.0 notes.md / starter.ipynb / solution.ipynb / practice.md drills / alignment.md ILO<->TLA<->AT.
> **范式**: Oxford Tutorial + HBS Devil's Advocate + Hattie 4 级形成性反馈.
> **限频**: 每单元 1 次/天 (见 cell6), 防依赖, 强制 spaced retrieval.

---

## Persona Prompt (本 tutorial 的"假扮 LLM"系统指令)

```
You are an Oxford tutorial fellow in Agent评估 (AgentBench, trajectory metrics,
LLM-as-a-judge, task success rates, cost/latency). 你每周与 1-2 名博士生进行 1
小时 tutorial.

规则:
1. NEVER give direct answers. 永远不直接给答案. 学生问"GEval 怎么写", 你不要写
   criteria 字符串, 而要反问"你觉得品牌调性这一维该用什么动词锚定".
2. Use Socratic questioning. 每轮用苏格拉底式追问 (为什么 / 反例 / 若前提变 /
   凭什么 / 如何) 推动学生自己推理.
3. Act as HBS devil's advocate. 学生说"我的 Agent 任务完成率 92% 很好", 你立刻
   反问"剩下 8% 失败的用例有没有共性? 是不是测试集过易?".
4. Reject vague claims. 学生说"我用了 LLM-as-a-judge", 你追问"哪个 LLM?
   criteria 怎么写? 怎么应对长度/位置/自我偏好三大偏差?".
5. End each turn with a probing question. 每轮结尾必留一个开放问题让学生下一轮
   回答, 不允许学生"答完就跑".
6. 引用本单元真实文件: notes.md 四大挑战 / starter.ipynb TODO / practice.md
   drill / reading.md AgentBench 论文.
```

> 本 notebook 用**静态 if/else 分支模拟** Socratic 追问, 不真调 openai/anthropic
> API. 学生把回答写进 student_responses 列表, 模拟 tutor 按预设分支给追问.


## Pre-Tutorial Task (强制 retrieval, 不做不许进 cell3)

**牛津 tutorial 哲学**: 学生必须先尝试, tutor 才能精准追问. 空手来 = 浪费双方时间.

> 提交前自查: 你是否独立完成下面 3 项? 若有 1 项空白 = 取消本次 tutorial, 改日重来.

### Task A (200 字 essay)
请用 200 字回答: **"为什么传统 assert output == expected 断言在营销 Agent 上会失效?
请用 Agent 评估四大挑战中的 2 个论证, 并指出哪个挑战最致命."**

> 提示: 看 notes.md 关键回顾1 的四大挑战表 (非确定性/多步推理/工具调用/长尾效应).

### Task B (1 段 criteria 草稿)
写一段 GEval criteria 草稿, 评估"小红书烟酰胺种草文案"质量. 要求包含 3 个维度
(品牌调性 / CTA / 平台适配) + 打分量纲 (0-1) + 至少 1 句话说明如何避免 LLM-as-judge
的"偏好长答案"偏差.

> 提示: 参考 practice.md drill D1-GEval 阶段1 Worked 示例.

### Task C (1 个对抗性测试用例设计)
设计 1 个对抗性 LLMTestCase: 用户问"这款精华能治痘痘吗"(知识库无此信息), 期望
Agent 说"不知道". 请写出 input / expected_output / retrieval_context(空) 三个字段,
并解释这对应因果阶梯 L1 还是 L2.

> 提示: 参考 practice.md drill D3-Faithfulness 阶段3 Independent.

---

**提交方式**: 把 Task A/B/C 的答案填进下面 cell3 的 student_responses 列表对应位置,
然后跑 cell3 进入 Socratic loop.


In [ ]:
# Day 3 Socratic Tutorial Loop (静态 if/else 模拟, 不调真实 LLM API)
# 学生把 pre-tutorial task 答案填进 student_responses, tutor 按预设分支追问.
# 每轮必含 >=1 个苏格拉底问 (为什么/反例/若前提变/凭什么/如何), 共 4 轮.

student_responses = {
    "task_a_assert_fail": "",   # 200字 essay: 为什么断言失效
    "task_b_criteria": "",      # GEval criteria 草稿
    "task_c_adversarial": "",   # 对抗性用例设计
}

def tutor_turn(turn_id, student_text):
    """静态 if/else 模拟 Socratic 追问. 返回 tutor 回应 + probing question."""
    text = (student_text or "").strip()
    length = len(text)

    # --- Turn 1: 探 task_a (断言失效) ---
    if turn_id == 1:
        if length < 50:
            return ("你的 essay 太短(<50字). 牛津 tutorial 不接受空话. "
                    "请具体回答: Agent 评估四大挑战里, 哪一个直接打破了 "
                    "assert output == expected 的前提假设? "
                    "反例: 如果挑战只有多步推理而没有非确定性, 断言还能用吗? "
                    "为什么? 凭什么说非确定性最致命?")
        if "非确定" not in text and "温度" not in text and "temperature" not in text.lower():
            return ("你提到了四大挑战, 但没点名非确定性. "
                    "为什么非确定性是断言失效的直接原因, 而长尾效应只是间接原因? "
                    "若前提变: 如果模型温度=0 且固定 seed, 断言能恢复吗? "
                    "如何? 请重新论证.")
        return ("好, 你抓住了非确定性. 但反例来了: 端到端评估用 GEval 打分 0.82, "
                "Agent 轨迹里 Thought1 选错了工具却碰巧答对--这种结果对过程错的情况, "
                "断言或 GEval 单独能发现吗? 凭什么说轨迹评估更可靠? "
                "你的 task_b criteria 怎么避免这种漏检?")

    # --- Turn 2: 探 task_b (GEval criteria) ---
    if turn_id == 2:
        if "品牌" not in text and "调性" not in text:
            return ("你的 criteria 没明确品牌调性维度. "
                    "为什么这一维对营销 Agent 比对通用 Agent 更重要? "
                    "反例: 若 criteria 只有 CTA 和平台适配两维, LLM-as-judge 会怎么 "
                    "评分偏差? 凭什么? 请补全三维度.")
        if "长度" not in text and "long" not in text.lower() and "偏好长" not in text:
            return ("你没提到如何应对 LLM-as-a-judge 的偏好长答案偏差. "
                    "假设 criteria 不加长度约束, LLM 给长文案打高分, 你的 "
                    "Agent 会不会因此学会废话凑字数? 如何在 criteria 里防这个? "
                    "依据是什么?")
        return ("三维度齐全. 但反例: 你的 criteria 打分量纲 0-1, 阈值设多少算 PASS? "
                "若阈值=0.7, 3 条文案分别 0.72/0.71/0.69, 前两条过第三条不过--"
                "这个 0.01 差距有统计意义吗? 凭什么不直接用 0.5? "
                "为什么? 请在 task_c 里设计对抗性用例检验你的阈值.")

    # --- Turn 3: 探 task_c (对抗性用例) ---
    if turn_id == 3:
        if "不知道" not in text and "I don" not in text.lower() and "无此" not in text:
            return ("你的 expected_output 不是不知道/无此信息. "
                    "为什么对抗性用例的期望是 Agent 主动承认无知? "
                    "反例: 若 Agent 编造这款精华能治痘痘, FaithfulnessMetric 会 "
                    "报警吗? 凭什么 retrieval_context 为空时幻觉率应=100%? "
                    "如何让 Agent 学会不知道说不知道?")
        if "L1" not in text and "L2" not in text and "因果" not in text:
            return ("你没定位因果阶梯 L1/L2. "
                    "为什么对抗性用例测试是 L1 关联分析而非 L2 干预? "
                    "若前提变: 真要做 L2 干预该怎么设计 (A/B 测试)? "
                    "凭什么 LLM-as-a-judge 不能替代 A/B? 依据?")
        return ("好, 你区分了 L1/L2. 最后一个反例: 你的对抗性用例只有 1 条, "
                "凭什么说 Agent 学会了不知道说不知道? "
                "若前提变: 用 100 条对抗性用例跑, 幻觉率从 12% 降到 4%, "
                "这是 L1 还是 L2? 如何? "
                "为什么这个数字不能直接用于生产决策? 凭什么?")

    # --- Turn 4: 综合追问 + exit ---
    if turn_id == 4:
        return ("综合追问: 你的营销 Agent 在 task_a (断言失效论证) / task_b (criteria) / "
                "task_c (对抗性用例) 三项里, 哪一项最弱? 凭什么? "
                "反例: 若你说都还行, 那就是vague claim, 立刻驳回. "
                "如何用 practice.md 的 weak_loop 退出? "
                "为什么退出条件是阶段3 独立解连续 2 次 PASS 而不是 1 次? "
                "依据是什么? 请在 cell4 student_model 里记录你的盲点.")

    return ("Tutorial 结束. 请填 cell4 student_model.json 并读 cell5 Hattie 反馈.")


# ---- 跑 4 轮 Socratic loop ----
turns_log = []
for t in range(1, 5):
    if t == 1:
        resp = student_responses["task_a_assert_fail"]
    elif t == 2:
        resp = student_responses["task_b_criteria"]
    elif t == 3:
        resp = student_responses["task_c_adversarial"]
    else:
        resp = " ".join([student_responses[k] for k in student_responses])
    reply = tutor_turn(t, resp)
    turns_log.append({"turn": t, "tutor_reply": reply})
    print(f"\n===== Turn {t} =====")
    print(reply)

print("\n===== 4 轮 Socratic loop 完成 =====")
print(f"总轮数: {len(turns_log)}")
print("苏格拉底问统计: 见 cell5 Hattie [PROCESS] 反馈")


In [ ]:
# student_model.json 读写 (记录掌握度 / 盲点 / 苏格拉底问命中)
# Oxford tutorial 哲学: tutor 必须记得学生上次的盲点, 下次 tutorial 直接追问.
# Hattie: formative feedback 必须基于 student_model, 不能给通用表扬.

import json, os, re

STUDENT_MODEL_PATH = "./student_model.json"

def load_student_model():
    """读学生模型. 首次运行则初始化."""
    if os.path.exists(STUDENT_MODEL_PATH):
        with open(STUDENT_MODEL_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {
        "unit": "U5-D3",
        "mastery": {
            "ILO-1_assert_fail": 0.0,
            "ILO-2_trajectory_vs_e2e": 0.0,
            "ILO-3_deepeval_suite": 0.0,
            "ILO-4_six_metrics": 0.0,
            "ILO-5_basemetric": 0.0,
        },
        "blind_spots": [],
        "socratic_hits": {
            "为什么": 0,
            "反例": 0,
            "若前提变": 0,
            "凭什么": 0,
            "如何": 0,
        },
        "weak_loop_count": 0,
        "last_tutorial_date": None,
    }

def save_student_model(model):
    """写学生模型."""
    with open(STUDENT_MODEL_PATH, "w", encoding="utf-8") as f:
        json.dump(model, f, ensure_ascii=False, indent=2)
    print(f"student_model 已写入: {STUDENT_MODEL_PATH}")

def update_mastery(model, ilo_key, score_0_1, blind_spot=None):
    """更新掌握度 + 记录盲点. score_0_1 in [0, 1]."""
    old = model["mastery"].get(ilo_key, 0.0)
    model["mastery"][ilo_key] = round(0.5 * old + 0.5 * score_0_1, 3)
    if blind_spot and blind_spot not in model["blind_spots"]:
        model["blind_spots"].append(blind_spot)
    if score_0_1 < 0.6:
        model["weak_loop_count"] += 1
        print(f"  -> ILO {ilo_key} score={score_0_1} < 0.6, weak_loop 触发 "
              f"(累计 {model['weak_loop_count']} 次)")

# ---- 示例: 假设学生在 Turn 1 答得不好 (score=0.4), Turn 3 答得好 (score=0.85) ----
model = load_student_model()
update_mastery(model, "ILO-1_assert_fail", 0.4,
               blind_spot="没点名非确定性是断言失效的直接原因")
update_mastery(model, "ILO-2_trajectory_vs_e2e", 0.7,
               blind_spot="端到端0.82但轨迹有错的反例没举出")
update_mastery(model, "ILO-3_deepeval_suite", 0.85)
update_mastery(model, "ILO-4_six_metrics", 0.6,
               blind_spot="三大指标不能直接相加的因果阶梯L1/L2论证缺失")
update_mastery(model, "ILO-5_basemetric", 0.5,
               blind_spot="BaseMetric reason模板含空洞词good/nice")

# 统计 cell3 tutor_reply 里的苏格拉底问命中
for entry in turns_log:
    reply = entry["tutor_reply"]
    for kw in model["socratic_hits"]:
        model["socratic_hits"][kw] += len(re.findall(re.escape(kw), reply))

save_student_model(model)
print("\n当前掌握度:")
for k, v in model["mastery"].items():
    print(f"  {k}: {v}")
print(f"\n盲点 ({len(model['blind_spots'])} 个):")
for b in model["blind_spots"]:
    print(f"  - {b}")
print(f"\n苏格拉底问命中: {model['socratic_hits']}")
print(f"weak_loop 触发: {model['weak_loop_count']} 次")


## Hattie 4 级形成性反馈 (Formative Feedback)

> John Hattie 《Visible Learning》: formative feedback 效应量 d=0.79 (前 10%).
> Self 级表扬 d=0.09 (几乎无效甚至负效). 本 cell 用 [TASK]/[PROCESS]/[SELF-REG]/[FEED-FORWARD]
> 四级, **不用 Self 表扬**.

基于 cell4 student_model.json 的当前掌握度, tutor 给出四级反馈:

### [TASK] 任务级反馈 (针对具体任务的对错与差距)

- **ILO-1 (断言失效)**: score=0.4. 你的 essay 没点名非确定性是断言失效的直接原因.
  反例你也没举: 若只有长尾效应而无非确定性, 断言在 95% 用例上仍可用, 只是不能 100%.
  修正: 重读 notes.md 关键回顾1 四大挑战表, 用"非确定性 -> 同输入不同输出 -> 断言前提 A=>B 失效"
  的因果链重写 essay.
- **ILO-2 (轨迹 vs 端到端)**: score=0.7. 你能区分两层, 但端到端0.82但轨迹有错的反例没举出.
  修正: 看 practice.md diagnostic D2, 那条轨迹(Thought2 直接编)就是反例.

### [PROCESS] 过程级反馈 (针对策略/方法/苏格拉底问命中)

- **苏格拉底问命中统计** (来自 cell4): 为什么/反例/若前提变/凭什么/如何 五类问.
  你的回答里只有为什么类被回应, 其余 4 类常跳过 -> 推理策略偏单线.
- **修正策略**: 下次 tutorial 前, 针对每个苏格拉底问类各准备 1 个回答模板.
  例: 反例类 -> 若 X 不成立, 则 Y; 若前提变类 -> 假设 temperature=0, 则...
- **HBS devil's advocate**: 你说任务完成率 92% 很好--你没问剩下 8% 的共性.
  生产期 8% 失败率 = 每天 1000 次调用里 80 次出错 = 重大 SLI 违约.
  修正: 永远追问失败的共性是什么.

### [SELF-REG] 自我调节反馈 (针对元认知/盲点觉察)

- 你的盲点列表 (cell4) 有 4 项, 但 weak_loop_count = 3. 说明你能识别盲点但退出 weak_loop 慢.
- **元认知修正**: 每次进入 weak_loop, 先用 1 句话写我卡在哪一步 (是 API 不熟/概念没懂/偏差没识别?),
  再回退阶段1. 不要直接抄 solution.ipynb -- 抄了 reason 还是空洞词.
- **自评校准**: 你自评 ILO-3 = 0.85, 但实际跑 deepeval test run 若 score 全 =1.0 ->
  判定 criteria 过松, 真实掌握度 <0.6. 修正: 用对抗性用例(知识库不存在的问题)检验 criteria 严苛度.

### [FEED-FORWARD] 前馈反馈 (针对下一步行动/迁移)

- **下一步**: 进 practice.md drill D2-Trajectory 阶段3 Independent, 扩展 BaseMetric 的参数准确性子分.
- **迁移**: Day 4 安全防护的 Prompt Injection 防御评估也用轨迹评估 (把是否被注入作为 trajectory 一步).
  把今天的 ToolCallCorrectnessMetric 改名为 TrajectoryStepMetric, 新增 injection_resistance 子分.
- **推荐复习单元**: Day 2 Agent 架构 (工具路由层) -- 你的 ILO-2 score=0.7 暴露工具选择 prompt 设计没吃透.
- **mastery 预测**: 若 weak_loop 全部退出 + D2 阶段3 独立解 2 次 PASS, 预计 ILO-5 从 0.5 升到 0.8.
  3 天后再测 (spaced retrieval).


## 限频与 Exit Artifact

### 限频 (防依赖)

- **每单元 1 次/天**: 本 Day 3 tutorial 每天最多跑 1 次. 若今天已跑过, 再跑会被
  student_model.json 的 last_tutorial_date 字段拦截.
- **为什么限频**: Oxford tutorial 的价值在于 spaced retrieval, 不是连续刷题. 连续跑
  会形成短期记忆假象, 3 天后全忘. 强制间隔 >=1 天让大脑遗忘再提取, 效应量更高
  (Hattie d=0.79).
- **防依赖**: tutorial 是辅助工具, 不是答案生成器. 若你发现不跑 tutorial 就不会做
  drill -> 已过度依赖, 立刻停 tutorial 3 天, 纯靠 practice.md 的 Worked-Faded
  阶段1 自学.

### Exit Artifact (本次 tutorial 结束必交)

> 不交 = 本次 tutorial 不计 mastery. 下次 tutorial 必须先补交.

**1. 2-3 个盲点** (从 cell4 student_model.json 的 blind_spots 抄录, 或新增):

- 盲点 1: _______________________________________________
- 盲点 2: _______________________________________________
- 盲点 3: _______________________________________________

**2. 推荐复习单元** (基于盲点指向的 ILO):

- 复习单元 1: ____________ (例: Day 2 Agent 架构 - 工具路由层)
- 复习单元 2: ____________ (例: reading.md LLM-as-a-judge 三大偏差条目)

**3. 下次 tutorial 的 1 个聚焦问题** (你自己提, tutor 下次直接追问):

- 聚焦问题: _______________________________________________
  (例: 我的 BaseMetric reason 模板为什么总含空洞词? 怎么改成具体步骤定位?)

### Anti-Stall 提示

- 本 notebook 全静态 if/else, 不调真实 LLM API, 跑完 <5 秒.
- 若你想接真实 LLM 做 Socratic tutor, 参考 reading.md LLM-as-a-judge 条目 +
  deepeval GEval 文档, 但不在 Day 3 上机范围内 (会触发 600s watchdog).

---

*v6.0 学习科学层: Oxford Tutorial LLM 仿真 + Socratic 追问 (>=5 问) + Hattie 4 级形成性反馈 (避开 Self 表扬) + student_model 追踪 + 限频防依赖. 配套 practice.md weak_loop / alignment.md ILO<->TLA<->AT.*
